In [ ]:
import json
import pandas as pd
from escher import Builder

# PATHS
MAP_JSON_PATH   = "c:/Users/HP/Downloads/RECON1.Glycolysis TCA PPP.json"
FLUX_CSV_PATH   = "c:/Users/HP/Desktop/Master's Project/INCAWrapper/Vehicle_fluxes_table.csv"
OUTPUT_HTML     = "c:/Users/HP/Desktop/Master's Project/Glycolysis_TCA_PPP_Flux_Map.html"

# Load Escher map 
with open(MAP_JSON_PATH, 'r', encoding='utf-8') as f:
    map_json_str = f.read()
map_data = json.loads(map_json_str)
if isinstance(map_data, list):
    map_data = map_data[0]
print(f"Loaded map: {map_data.get('map_name', 'Unknown')}")

# Load net fluxes from CSV 
rows = []
with open(FLUX_CSV_PATH, 'r') as f:
    lines = f.readlines()
for line in lines[1:]:
    if line.startswith('Net flux'):
        parts = line.split(',')
        if len(parts) >= 4:
            try:
                rows.append({
                    'id':  parts[1].strip(),
                    'val': float(parts[3].strip())
                })
            except ValueError:
                pass

df       = pd.DataFrame(rows)
flux_val = df.set_index('id')['val'].to_dict()
print(f"Loaded {len(flux_val)} net flux values.")

# Build reaction_fluxes dictionary
reaction_fluxes = {}

# Direct 1-to-1 mappings 
direct = {
    # Glycolysis
    'R2 net':  'PGI',        # G6P <-> F6P
    'R3':      'PFK',        # F6P -> FBP
    'R4 net':  'FBA',        # FBP <-> DHAP + GAP
    'R5 net':  'TPI',        # DHAP <-> GAP
    'R8':      'PYK',        # PEP -> Pyr
    'R9 net':  'LDH_L',      # Pyr <-> Lac
    'R10':     'L_LACt2r',   # Lac -> Lac.x (extracellular)
    'R13':     'GLYCOGENt',  # Glycogen -> G6P (may not be on map)

    # Pyruvate handling
    'R19':     'PYRt2m',     # Pyr.c -> Pyr.m
    'R24':     'PDHm',       # Pyr.m -> AcCoA.m + CO2

    # PPP
    'R16 net': 'TKT1',       # R5P + R5P <-> S7P + GAP
    'R17 net': 'TALA',       # S7P + GAP <-> F6P + E4P
    'R18 net': 'TKT2',       # R5P + E4P <-> F6P + GAP

    # TCA cycle
    'R33':     'CSm',        # AcCoA.m + Oac.m -> Cit.m
    'R36 net': 'SUCD1m',     # Suc <-> Fum.m
    'R37 net': 'FUMm',       # Fum.m <-> Mal.m
    'R38 net': 'MDHm',       # Mal.m <-> Oac.m

    # Cytoplasmic reactions
    'R40 net': 'MDH',        # Mal.c <-> Oac.c
    'R41 net': 'ASPTA',      # Oac.c <-> Asp.c
    'R39 net': 'ASPTAm',     # Oac.m <-> Asp.m

    # Transport
    'R44 net': 'AKGMALtm',   # Mal.c <-> Mal.m (malate-alphaKG transporter)

    # Reductive TCA / citrate shuttle
    'R46':     'CITtam',     # Cit.m -> Cit.c
    'R48':     'ACLYc',      # Cit.c -> AcCoA.c + Oac.c (ATP citrate lyase)
}

for inca_id, bigg_id in direct.items():
    if inca_id in flux_val:
        reaction_fluxes[bigg_id] = float(flux_val[inca_id])

# Split reactions (divide equally between constituent enzymes) -> Handling for combined linear reactions

splits = {
    # R1 = glucose transport (GLCt1) + hexokinase (HEX1)
    'R1':      ['GLCt1', 'HEX1'],

    # R6 = GAPDH (GAPD) + phosphoglycerate kinase (PGK)
    'R6 net':  ['GAPD', 'PGK'],

    # R7 = phosphoglycerate mutase (PGM) + enolase (ENO)
    'R7 net':  ['PGM', 'ENO'],

    # R15 = G6PDH2r + PGL + GND + RPI  (oxidative PPP: 4 steps)
    'R15':     ['G6PDH2r', 'PGL', 'GND', 'RPI'],

    # R34 = aconitase (ACONTm) + isocitrate dehydrogenase (ICDHxm)
    'R34':     ['ACONTm', 'ICDHxm'],

    # R35 = alpha-ketoglutarate dehydrogenase (AKGDm) + succinyl-CoA synthetase (SUCOAS1m)
    'R35':     ['AKGDm', 'SUCOAS1m'],

    # R45 + R28 combined = aspartate-glutamate carrier (ASPGLUm)
    # R45 net: Asp.m <-> Asp.c   R28 net: Glu.c <-> Glu.m
    # Use R45 as the representative flux for this transporter
    'R45 net': ['ASPGLUm'],
}

for inca_id, bigg_ids in splits.items():
    if inca_id in flux_val:
        split_val = float(flux_val[inca_id]) / len(bigg_ids)
        for bigg_id in bigg_ids:
            reaction_fluxes[bigg_id] = split_val
        print(f"  Split {inca_id} ({flux_val[inca_id]:.4f}) "
              f"→ {len(bigg_ids)} × {split_val:.4f}  "
              f"[{', '.join(bigg_ids)}]")

# R12 (Pyr.m -> Ala): alanine aminotransferase, mitochondrial
# Note: R11 (Pyr.c -> Ala) ≈ 0, so R12 dominates
if 'R12' in flux_val:
    reaction_fluxes['ALATAm'] = float(flux_val['R12'])

print(f"\nTotal reactions mapped to Escher: {len(reaction_fluxes)}")
print("\nReaction fluxes:")
for k, v in sorted(reaction_fluxes.items()):
    print(f"  {k:20s}  {v:+.6f}")

# Build Escher map
builder = Builder(
    map_json          = map_json_str,
    reaction_data     = reaction_fluxes,
    reaction_styles   = ['color', 'size', 'text'],
    reaction_scale    = [
        {'type': 'min',    'color': '#ff3333', 'size': 6},
        {'type': 'zero',   'color': '#cccccc', 'size': 4},
        {'type': 'mean',   'color': '#aaaaff', 'size': 12},
        {'type': 'max',    'color': '#3333ff', 'size': 24},
    ],
    reaction_no_data_color = '#f0f0f0',
    show_gene_reaction_rules = False,
)

#Save
builder.save_html(OUTPUT_HTML)
print(f"\nFlux map saved as '{OUTPUT_HTML}'")

# Display in Jupyter
builder

Loaded map: RECON1.Glycolysis TCA PPP
Loaded 68 net flux values.
  Split R1 (5.7500) → 2 × 2.8750  [GLCt1, HEX1]
  Split R6 net (14.8386) → 2 × 7.4193  [GAPD, PGK]
  Split R7 net (14.8386) → 2 × 7.4193  [PGM, ENO]
  Split R15 (0.0001) → 4 × 0.0000  [G6PDH2r, PGL, GND, RPI]
  Split R34 (0.0000) → 2 × 0.0000  [ACONTm, ICDHxm]
  Split R35 (-0.0011) → 2 × -0.0006  [AKGDm, SUCOAS1m]
  Split R45 net (-0.0058) → 1 × -0.0058  [ASPGLUm]

Total reactions mapped to Escher: 39

Reaction fluxes:
  ACLYc                 +0.260387
  ACONTm                +0.000010
  AKGDm                 -0.000559
  AKGMALtm              +0.001259
  ALATAm                +8.089771
  ASPGLUm               -0.005821
  ASPTA                 +0.803058
  ASPTAm                -0.005821
  CITtam                +0.005915
  CSm                   +0.005935
  ENO                   +7.419302
  FBA                   +7.419289
  FUMm                  -0.001119
  G6PDH2r               +0.000018
  GAPD                  +7.419302
  

Builder(reaction_data={'PGI': 7.419241314373385, 'PFK': 7.419289495635126, 'FBA': 7.419289495635126, 'TPI': 7.…